## For Statues
### This was for Statues which all are searchable PDFs

In [ ]:
!pip install -q pymupdf4llm qdrant-client llama-index-llms-groq llama-index-vector-stores-qdrant llama-index-embeddings-openai

In [ ]:
import os
import pymupdf4llm
from google.colab import userdata
from qdrant_client import QdrantClient, models
from llama_index.core import Document, VectorStoreIndex
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core import StorageContext
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.groq import Groq

# Environment Setup
os.environ['OPENAI_API_KEY'] = userdata.get("OPENAI_API_KEY")
os.environ['GROQ_API_KEY'] = userdata.get("GROQ_API_KEY")
QDRANT_URL = userdata.get("QDRANT_URL")
QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")

In [ ]:
# Create a 'data' directory and upload your PDF files there.
DATA_DIR = "data"
if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR)

documents = []
for filename in os.listdir(DATA_DIR):
    if filename.endswith(".pdf"):
        pdf_path = os.path.join(DATA_DIR, filename)
        print(f"Processing {pdf_path}...")
        # Extract text as Markdown for better structural representation
        md_text = pymupdf4llm.to_markdown(pdf_path)

        # Create a LlamaIndex Document object
        doc = Document(
            text=md_text,
            metadata={"file_name": filename}
        )
        documents.append(doc)

print(f"\nSuccessfully loaded and processed {len(documents)} PDF documents.")

Processing data/Federal Ziauddin University Act, 2024.pdf...
Processing data/West Pakistan Urban Immovable Property Tax Act, 1958.pdf...
Processing data/Claims for Maintenance (Recovery Abroad) Ordinance, 1959.pdf...
Processing data/Life Insurance (Nationalisation) Order, 1972.pdf...
Processing data/Pakistan Penal Code (PPC),1860 (Under Review).pdf...
Processing data/Drugs and Medicines (lndemnity) Act, 1957.pdf...
Processing data/Inland Mechanically Propelled Vessels Act, 1917.pdf...
Processing data/Motion Pictures Ordinance, 1979.pdf...
Processing data/Peaceful Assembly and Public Order Act, 2024.pdf...
Processing data/Islamabad High Court Act, 2010.pdf...
Processing data/Pakistan Water and Power Development Authority Act, 1958.pdf...
Processing data/Jute (Repeal) Ordinance, 1983.pdf...
Processing data/Central Depositories Act, 1997.pdf...
Processing data/West Pakistan Epidemic Diseases Act, 1958.pdf...
Processing data/House Building Finance Corporation Act, 1952(Repeal by act XXV of

In [ ]:
documents[99].text

'# **THE COTTON CLOTH AND YARN (CONTRACTS) ORDINANCE,**\n\nPage **1** of **2**\n\n\n## **THE COTTON CLOTH AND YARN (CONTRACTS) ORDINANCE, 1944.** ORDINANCE No. II OF 1944\n\n[ _13_ _[th]_ _January, 1944_ ]\n\n\n_An Ordinance to regulate the prices which may be charged in sales of cotton cloth and yarn under_\n\n\n_contracts._\n\n\nDate: 01-07-2024\n\n\n1 Subs. by the Central Laws (Statute Reform) Ordinance No. XXI of 1960, s. 3 and 2nd Sch. ( _with effect from the 14th October, 1955_ ).\n\n\nPage **2** of **2**\n\n\n'

In [ ]:
# Qdrant Client
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

collection_name = "general"

# Create collection with binary quantization if it doesn't exist
if not client.collection_exists(collection_name=collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(
        size=1536, # For OpenAI embedding model text-embedding-3-small
        distance=models.Distance.COSINE,
        on_disk=True  # Move original vectors to disk
    ),
    quantization_config=models.BinaryQuantization(
        binary=models.BinaryQuantizationConfig(
            always_ram=True  # Store only quantized vectors in RAM
        )
    )
    )
    print(f"Collection '{collection_name}' created.")
else:
    print(f"Collection '{collection_name}' already exists.")

Collection 'general' created.


In [ ]:
# Use the updated GoogleGenaiEmbedding class
embed_model = OpenAIEmbedding(
    model_name="text-embedding-3-small"
    )

# Qdrant Vector Store
vector_store = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
)

In [ ]:
# Storage Context to link the vector store
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# Create the index
# This will embed the documents and store them in Qdrant
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
    embed_model=embed_model,
    show_progress=True
)

print("\nIndexing complete.")

Parsing nodes:   0%|          | 0/1003 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/237 [00:00<?, ?it/s]


Indexing complete.


In [ ]:
from llama_index.core import Document, VectorStoreIndex, get_response_synthesizer
from llama_index.core.retrievers import VectorIndexRetriever

# LLM for response generation
llm = Groq(model="llama-3.3-70b-versatile")

# Retriever to fetch relevant documents from the index
retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=5,
)

# Response Synthesizer to generate a response from the retrieved context
response_synthesizer = get_response_synthesizer(
    llm=llm,
    response_mode="compact",
)

# Assemble the query engine
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer,
)

print("Query engine is ready.")

Query engine is ready.


In [ ]:
# The embedding model for the query should have the 'retrieval_query' task type
query_engine.retriever.embed_model = embed_model

# Now, query your data
query_text = "what is the summary of constitution of pakistan"
response = query_engine.query(query_text)

print("Query:", query_text)
print("\nResponse:")
print(response)

Query: what is the summary of constitution of pakistan

Response:
The Constitution of Pakistan is a comprehensive document that outlines the fundamental principles, structures, and powers of the government in Pakistan. It establishes Pakistan as a Federal Republic, with Islam as the state religion, and guarantees various fundamental rights to its citizens, including equality, justice, and freedom of speech and religion. The Constitution also provides for the independence of the judiciary, the protection of minority rights, and the promotion of social justice and economic well-being in Pakistan. It divides the country into provinces and territories, with the federal government having certain powers and the provinces having autonomy in other areas. The Constitution also sets out the principles of policy, which include the promotion of Islamic values, the protection of the environment, and the promotion of international peace, with the aim of creating a just and equitable society in Pakis